# 🧬 Ridge and Lasso for High-Dimensional Genomic Data
## Regularization, feature selection, and generalization when $p \gg n$

### Overview

Modern genomic studies often measure hundreds or thousands of genes in far fewer patients. This creates a **high-dimensional** learning problem: the number of features $p$ may be larger than the number of samples $n$. In that setting, ordinary least squares can fit the training data extremely well while producing unstable coefficients and poor predictions for new patients.

This tutorial builds a controlled gene-expression experiment and follows one question:

~~~text
High-dimensional data → Unstable least squares → Regularization → Cross-validation → Scientific interpretation
~~~

The response is a simulated continuous drug-sensitivity score. Because the data are simulated, we know which genes generated the response and can evaluate both prediction and feature recovery. The simulation is a teaching model—not evidence about a real drug or biological mechanism.

### Learning Objectives

By the end of this tutorial, you will be able to:

1. explain why ordinary least squares is unstable when $p>n$ and genes are correlated;
2. interpret the $L_2$ penalty in Ridge regression and the $L_1$ penalty in Lasso regression;
3. implement OLS, Ridge, Lasso, and $K$-fold cross-validation with NumPy;
4. select a regularization strength without using the test set;
5. compare prediction error, coefficient shrinkage, and sparsity;
6. explain why a selected gene is not automatically a causal biomarker.

### Tutorial Roadmap

| Part | Topic | Main output |
| --- | --- | --- |
| 0 | Controlled genomic experiment | Correlated gene-expression data |
| 1 | Ordinary least squares | Overfitting and coefficient instability |
| 2 | Ridge regression | Stable, dense coefficients |
| 3 | Lasso regression | Sparse coefficients and feature selection |
| 4 | Model comparison | Test RMSE and scientific interpretation |

### Runtime and Prerequisites

- **Hardware:** CPU only
- **Expected runtime:** under one minute on a typical laptop
- **Libraries:** NumPy, pandas, Matplotlib
- **Prerequisites:** basic Python, vectors and matrices, train/test splits

> Important: the test set is used only once, after all model and hyperparameter choices are complete.

In [ ]:
# Part 0.1 — Environment setup
# ============================================================
# Only standard scientific Python libraries are used. A fixed
# random seed makes every simulated dataset and split reproducible.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.titlesize": 11,
    "axes.labelsize": 10,
})

print("Environment setup completed.")
print("NumPy version:", np.__version__)
print("No external data download or GPU is required.")

# Part 0 — Build a Controlled Genomic Experiment

Suppose each row of $X\in\mathbb{R}^{n\times p}$ is one patient and each column is the standardized expression of one gene. We assume a linear data-generating process:

$$
y_i = \beta_0 + x_i^\top\beta + \varepsilon_i,
\qquad
\varepsilon_i\sim\mathcal{N}(0,\sigma^2).
$$

Here $y_i$ is a continuous drug-sensitivity score. Only a small number of entries in $\beta$ are nonzero, but genes are organized into correlated pathway-like blocks. Consequently, a non-causal proxy gene may contain nearly the same predictive information as a causal gene.

We simulate:

- $n=120$ patients and $p=200$ genes, so $p>n$;
- 8 correlated pathway blocks;
- 8 signal genes that directly generate the response;
- Gaussian measurement noise.

The train/test split is performed **before** standardization. This prevents the test patients from influencing the fitted preprocessing parameters.

In [ ]:
# Part 0.2 — Simulate correlated gene-expression measurements
# ============================================================

n_samples = 120
n_genes = 200
n_pathways = 8
genes_per_pathway = 15

# Each latent pathway affects a block of genes. Genes in the same
# block therefore have correlated expression, as often happens in biology.
pathway_activity = rng.normal(size=(n_samples, n_pathways))
X = rng.normal(size=(n_samples, n_genes))

for pathway in range(n_pathways):
    start = pathway * genes_per_pathway
    stop = start + genes_per_pathway
    shared_signal = pathway_activity[:, [pathway]]

    # A small gene-specific noise term keeps correlated genes from
    # becoming exact copies of one another.
    X[:, start:stop] = shared_signal + 0.35 * rng.normal(
        size=(n_samples, genes_per_pathway)
    )

# Select one direct signal gene from each pathway block.
true_gene_indices = np.arange(n_pathways) * genes_per_pathway
true_beta = np.zeros(n_genes)
true_beta[true_gene_indices] = np.array([3.0, -2.7, 2.4, -2.1, 1.9, -1.7, 1.5, -1.3])

# The response contains a learnable linear signal plus irreducible noise.
noise_sd = 1.5
y = 2.0 + X @ true_beta + rng.normal(scale=noise_sd, size=n_samples)

# Shuffle once, then reserve 25% of patients for final evaluation.
indices = rng.permutation(n_samples)
n_train = int(0.75 * n_samples)
train_idx, test_idx = indices[:n_train], indices[n_train:]

X_train_raw, X_test_raw = X[train_idx], X[test_idx]
y_train_raw, y_test_raw = y[train_idx], y[test_idx]

print(f"Training patients: {X_train_raw.shape[0]}")
print(f"Test patients:     {X_test_raw.shape[0]}")
print(f"Measured genes:    {X_train_raw.shape[1]}")
print(f"Direct signal genes: {len(true_gene_indices)}")
print(f"High-dimensional? p > n_train is {n_genes > n_train}")

In [ ]:
# Part 0.3 — Standardize using training statistics only
# ============================================================
# Regularization depends on coefficient scale. If one gene is measured
# in large units and another in small units, an equal penalty would not
# treat them fairly. We therefore center and scale every gene.

gene_mean = X_train_raw.mean(axis=0)
gene_std = X_train_raw.std(axis=0, ddof=0)
gene_std[gene_std == 0] = 1.0  # Defensive guard for constant features.

X_train = (X_train_raw - gene_mean) / gene_std
X_test = (X_test_raw - gene_mean) / gene_std

# Center y so that the models can focus on gene coefficients. The
# intercept is restored by adding y_mean to every final prediction.
y_mean = y_train_raw.mean()
y_train = y_train_raw - y_mean
y_test = y_test_raw - y_mean

# Inspect correlation inside and outside a pathway block.
within_pathway_corr = np.corrcoef(X_train[:, 0], X_train[:, 1])[0, 1]
between_pathway_corr = np.corrcoef(X_train[:, 0], X_train[:, 30])[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(np.corrcoef(X_train[:, :45], rowvar=False), vmin=-1, vmax=1, cmap="coolwarm")
axes[0].set_title("Correlation among the first 45 genes")
axes[0].set_xlabel("Gene index")
axes[0].set_ylabel("Gene index")

axes[1].bar(["Patients (train)", "Genes"], [X_train.shape[0], X_train.shape[1]], color=["#4C78A8", "#F58518"])
axes[1].set_title("The high-dimensional setting")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

print(f"Correlation within one pathway:  {within_pathway_corr:.3f}")
print(f"Correlation across two pathways: {between_pathway_corr:.3f}")

assert X_train.shape[1] > X_train.shape[0]
assert np.allclose(X_train.mean(axis=0), 0.0, atol=1e-12)
assert np.allclose(X_train.std(axis=0), 1.0, atol=1e-12)
print("Checkpoint passed: the training design is standardized and p > n.")

# Part 1 — Why Ordinary Least Squares Struggles

Ordinary least squares (OLS) minimizes the residual sum of squares:

$$
\hat\beta_{\mathrm{OLS}}
=\arg\min_\beta\frac{1}{2n}\|y-X\beta\|_2^2.
$$

When $p<n$ and $X^\top X$ is invertible, the familiar solution is

$$
\hat\beta=(X^\top X)^{-1}X^\top y.
$$

Here $p>n$, so $\operatorname{rank}(X)\le n<p$ and $X^\top X$ cannot have full rank. There are directions in coefficient space that the training data cannot distinguish. NumPy's `lstsq` returns one minimum-norm solution, but that choice may still be sensitive to small changes in the patients included in training.

We evaluate prediction with root mean squared error:

$$
\operatorname{RMSE}(y,\hat y)
=\sqrt{\frac{1}{m}\sum_{i=1}^{m}(y_i-\hat y_i)^2}.
$$

A large gap between training and test RMSE is evidence that the model has fit patterns that do not generalize.

In [ ]:
# Part 1.1 — Fit and diagnose ordinary least squares
# ============================================================

def rmse(y_true, y_pred):
    '''Return root mean squared prediction error.'''
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def fit_ols(X_fit, y_fit):
    '''Return NumPy's minimum-norm least-squares solution.'''
    coefficients, *_ = np.linalg.lstsq(X_fit, y_fit, rcond=None)
    return coefficients


ols_coef = fit_ols(X_train, y_train)
ols_train_pred = X_train @ ols_coef
ols_test_pred = X_test @ ols_coef

print(f"OLS training RMSE: {rmse(y_train, ols_train_pred):.3f}")
print(f"OLS test RMSE:     {rmse(y_test, ols_test_pred):.3f}")
print(f"Coefficient L2 norm: {np.linalg.norm(ols_coef):.2f}")
print(f"Numerical rank of X: {np.linalg.matrix_rank(X_train)} / {X_train.shape[1]} columns")

# Because p > n, X has a non-empty null space. Moving the coefficient
# vector along a null-space direction does not change X @ beta at all.
# This constructs a second, very different model with the same training fit.
_, _, Vt_full = np.linalg.svd(X_train, full_matrices=True)
null_direction = Vt_full[-1]
assert np.linalg.norm(X_train @ null_direction) < 1e-10

ols_alternative = ols_coef + 10.0 * null_direction
alternative_train_pred = X_train @ ols_alternative
alternative_test_pred = X_test @ ols_alternative

print(f"Distance between two OLS coefficient vectors: {np.linalg.norm(ols_coef - ols_alternative):.2f}")
print(f"Maximum training-prediction difference: {np.max(np.abs(ols_train_pred - alternative_train_pred)):.2e}")
print(f"RMSE between their test predictions: {rmse(ols_test_pred, alternative_test_pred):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_train, ols_train_pred, alpha=0.7, label="Train")
axes[0].scatter(y_test, ols_test_pred, alpha=0.8, label="Test")
limit = np.max(np.abs(np.r_[y_train, y_test, ols_train_pred, ols_test_pred]))
axes[0].plot([-limit, limit], [-limit, limit], "k--", linewidth=1)
axes[0].set(xlabel="Observed centered response", ylabel="Predicted response", title="OLS predictions")
axes[0].legend()

axes[1].scatter(ols_coef, ols_alternative, s=18, alpha=0.65)
coef_limit = np.max(np.abs(np.r_[ols_coef, ols_alternative]))
axes[1].plot([-coef_limit, coef_limit], [-coef_limit, coef_limit], "k--", linewidth=1)
axes[1].set(xlabel="Minimum-norm OLS", ylabel="Alternative OLS solution", title="Same training fit, different coefficients")

plt.tight_layout()
plt.show()

assert rmse(y_train, ols_train_pred) < rmse(y_test, ols_test_pred)
assert np.allclose(ols_train_pred, alternative_train_pred)
print("Checkpoint passed: OLS fits training data better than unseen patients.")

# Part 2 — Ridge Regression: Stabilize Rather Than Select

Ridge regression adds a squared $L_2$ penalty:

$$
\hat\beta_{\mathrm{Ridge}}
=\arg\min_\beta
\left[
\frac{1}{2n}\|y-X\beta\|_2^2
+\frac{\lambda}{2}\|\beta\|_2^2
\right].
$$

Setting the gradient to zero gives

$$
\hat\beta_{\mathrm{Ridge}}
=\left(X^\top X+n\lambda I\right)^{-1}X^\top y.
$$

For every $\lambda>0$, the added $n\lambda I$ makes the system invertible even when $p>n$. Larger $\lambda$ shrinks coefficients more strongly:

- $\lambda\rightarrow0$: approach the least-squares fit;
- moderate $\lambda$: reduce variance and stabilize prediction;
- very large $\lambda$: shrink toward zero and underfit.

Ridge usually keeps **all** genes with small coefficients. This is helpful when many correlated genes share predictive information.

### Probabilistic interpretation

Under Gaussian observation noise and an independent Gaussian prior

$$
\beta_j\sim\mathcal N(0,\tau^2),
$$

the maximum a posteriori estimate has the Ridge form. Regularization therefore encodes the assumption that extremely large effects are less plausible before observing the data.

In [ ]:
# Part 2.1 — Implement Ridge and trace the regularization path
# ============================================================

def fit_ridge(X_fit, y_fit, lam):
    '''Fit Ridge for objective RSS/(2n) + lam*||beta||_2^2/2.

    The SVD implementation is numerically stable and efficient when p > n.
    If X = U diag(s) V^T, each singular direction is shrunk by
    s / (s^2 + n*lam).
    '''
    n = X_fit.shape[0]
    U, singular_values, Vt = np.linalg.svd(X_fit, full_matrices=False)
    shrinkage = singular_values / (singular_values**2 + n * lam)
    return Vt.T @ (shrinkage * (U.T @ y_fit))


ridge_lambdas = np.logspace(-4, 2, 35)
ridge_path = np.column_stack([
    fit_ridge(X_train, y_train, lam) for lam in ridge_lambdas
])

ridge_train_rmse = np.array([
    rmse(y_train, X_train @ ridge_path[:, i])
    for i in range(len(ridge_lambdas))
])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogx(ridge_lambdas, ridge_path[true_gene_indices].T, linewidth=1.5)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set(xlabel=r"$\lambda$", ylabel="Standardized coefficient", title="Ridge paths for the 8 signal genes")

axes[1].semilogx(ridge_lambdas, ridge_train_rmse, color="#4C78A8")
axes[1].set(xlabel=r"$\lambda$", ylabel="Training RMSE", title="Training error alone favors weak regularization")

plt.tight_layout()
plt.show()

print("At large lambda, coefficients move smoothly toward zero.")
print("Ridge coefficients exactly equal to zero:", np.sum(ridge_path[:, -1] == 0))

## Select $\lambda$ with Cross-Validation

Choosing the value that minimizes training error would favor almost no regularization. Instead, $K$-fold cross-validation repeatedly trains on $K-1$ folds and evaluates on the remaining fold:

$$
\operatorname{CV}(\lambda)
=\frac{1}{K}\sum_{k=1}^{K}
\operatorname{MSE}\left(y^{(k)},\hat y_{-k}^{(k)}(\lambda)\right).
$$

The subscript $-k$ means that fold $k$ was not used to fit the model. We select $\lambda$ using only the training set. The untouched test set remains an unbiased final check.

In [ ]:
# Part 2.2 — Five-fold cross-validation for Ridge
# ============================================================

def make_folds(n_rows, n_folds=5, seed=SEED):
    '''Return reproducible arrays of validation-row indices.'''
    local_rng = np.random.default_rng(seed)
    shuffled = local_rng.permutation(n_rows)
    return np.array_split(shuffled, n_folds)


def cross_validate(X_cv, y_cv, parameter_values, fit_function, n_folds=5):
    '''Calculate validation MSE for each candidate regularization value.'''
    folds = make_folds(len(y_cv), n_folds=n_folds)
    mean_mse = []
    standard_error = []

    for value in parameter_values:
        fold_mse = []
        for validation_rows in folds:
            training_mask = np.ones(len(y_cv), dtype=bool)
            training_mask[validation_rows] = False

            coefficients = fit_function(
                X_cv[training_mask], y_cv[training_mask], value
            )
            predictions = X_cv[validation_rows] @ coefficients
            fold_mse.append(np.mean((y_cv[validation_rows] - predictions) ** 2))

        mean_mse.append(np.mean(fold_mse))
        standard_error.append(np.std(fold_mse, ddof=1) / np.sqrt(n_folds))

    return np.array(mean_mse), np.array(standard_error)


ridge_cv_mse, ridge_cv_se = cross_validate(
    X_train, y_train, ridge_lambdas, fit_ridge, n_folds=5
)
best_ridge_lambda = ridge_lambdas[np.argmin(ridge_cv_mse)]
ridge_coef = fit_ridge(X_train, y_train, best_ridge_lambda)

plt.figure(figsize=(7, 4))
plt.semilogx(ridge_lambdas, ridge_cv_mse, color="#4C78A8")
plt.fill_between(
    ridge_lambdas,
    ridge_cv_mse - ridge_cv_se,
    ridge_cv_mse + ridge_cv_se,
    color="#4C78A8",
    alpha=0.2,
    label="± 1 standard error",
)
plt.axvline(best_ridge_lambda, color="#E45756", linestyle="--", label=f"selected λ = {best_ridge_lambda:.3g}")
plt.xlabel(r"$\lambda$")
plt.ylabel("Cross-validated MSE")
plt.title("Choose Ridge regularization without viewing the test set")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Selected Ridge lambda: {best_ridge_lambda:.4g}")
print(f"Ridge coefficient L2 norm: {np.linalg.norm(ridge_coef):.2f}")

# Part 3 — Lasso Regression: Shrink and Select

Lasso replaces the squared penalty with an $L_1$ penalty:

$$
\hat\beta_{\mathrm{Lasso}}
=\arg\min_\beta
\left[
\frac{1}{2n}\|y-X\beta\|_2^2
+\lambda\|\beta\|_1
\right],
\qquad
\|\beta\|_1=\sum_{j=1}^{p}|\beta_j|.
$$

Unlike the smooth Ridge penalty, $|\beta_j|$ has a corner at zero. That geometry permits an optimum with many coefficients exactly equal to zero. Lasso therefore performs estimation and feature selection together.

We implement cyclic coordinate descent. Holding all other coefficients fixed, define the partial residual

$$
r_j=y-\sum_{k\ne j}X_k\beta_k.
$$

The update for a standardized feature is a soft-threshold operation:

$$
\beta_j\leftarrow
\frac{S_\lambda\left(X_j^\top r_j/n\right)}{X_j^\top X_j/n},
\qquad
S_\lambda(z)=\operatorname{sign}(z)\max(|z|-\lambda,0).
$$

Soft thresholding explains the exact zeros: correlations smaller than the threshold are removed.

In [ ]:
# Part 3.1 — Implement Lasso with coordinate descent
# ============================================================

def soft_threshold(value, threshold):
    '''Move a scalar toward zero and set small values exactly to zero.'''
    return np.sign(value) * max(abs(value) - threshold, 0.0)


def fit_lasso(X_fit, y_fit, lam, initial=None, max_iter=2000, tol=1e-7):
    '''Fit Lasso for objective RSS/(2n) + lam*||beta||_1.

    Each loop updates one coefficient at a time. The residual is updated
    in place, avoiding a complete prediction recomputation for every gene.
    '''
    n, p = X_fit.shape
    beta = np.zeros(p) if initial is None else initial.copy()
    residual = y_fit - X_fit @ beta
    feature_scale = np.sum(X_fit**2, axis=0) / n

    for _ in range(max_iter):
        largest_change = 0.0

        for gene in range(p):
            old_value = beta[gene]

            # Add gene j back to obtain the partial residual r_j.
            residual += X_fit[:, gene] * old_value
            correlation = X_fit[:, gene] @ residual / n
            new_value = soft_threshold(correlation, lam) / feature_scale[gene]

            # Remove the updated contribution before moving to the next gene.
            residual -= X_fit[:, gene] * new_value
            beta[gene] = new_value
            largest_change = max(largest_change, abs(new_value - old_value))

        if largest_change < tol:
            break

    return beta


# lambda_max is the smallest lambda that makes every coefficient zero.
lambda_max = np.max(np.abs(X_train.T @ y_train)) / len(y_train)
lasso_lambdas_desc = np.geomspace(lambda_max, lambda_max * 0.01, 30)

# Warm starts make neighboring path solutions much faster to compute.
lasso_path_desc = []
warm_start = np.zeros(n_genes)
for lam in lasso_lambdas_desc:
    warm_start = fit_lasso(X_train, y_train, lam, initial=warm_start)
    lasso_path_desc.append(warm_start.copy())

lasso_path_desc = np.column_stack(lasso_path_desc)
nonzero_counts = np.sum(np.abs(lasso_path_desc) > 1e-8, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogx(lasso_lambdas_desc, lasso_path_desc[true_gene_indices].T, linewidth=1.5)
axes[0].invert_xaxis()
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set(xlabel=r"$\lambda$ (strong → weak)", ylabel="Standardized coefficient", title="Lasso paths for the 8 signal genes")

axes[1].semilogx(lasso_lambdas_desc, nonzero_counts, marker="o", markersize=3)
axes[1].invert_xaxis()
axes[1].set(xlabel=r"$\lambda$ (strong → weak)", ylabel="Number of nonzero coefficients", title="Sparsity changes with regularization")

plt.tight_layout()
plt.show()

assert nonzero_counts[0] == 0
print("Checkpoint passed: lambda_max produces an all-zero Lasso model.")

In [ ]:
# Part 3.2 — Cross-validate Lasso and inspect selected genes
# ============================================================
# Use a smaller candidate grid for cross-validation to keep this teaching
# notebook fast. The same five fold assignments are used for every lambda.

lasso_lambdas = np.geomspace(lambda_max, lambda_max * 0.01, 18)
lasso_cv_mse, lasso_cv_se = cross_validate(
    X_train, y_train, lasso_lambdas, fit_lasso, n_folds=5
)
best_lasso_lambda = lasso_lambdas[np.argmin(lasso_cv_mse)]
lasso_coef = fit_lasso(X_train, y_train, best_lasso_lambda)

selected_genes = np.flatnonzero(np.abs(lasso_coef) > 1e-8)
recovered_direct_genes = np.intersect1d(selected_genes, true_gene_indices)

plt.figure(figsize=(7, 4))
plt.semilogx(lasso_lambdas, lasso_cv_mse, color="#54A24B")
plt.fill_between(
    lasso_lambdas,
    lasso_cv_mse - lasso_cv_se,
    lasso_cv_mse + lasso_cv_se,
    color="#54A24B",
    alpha=0.2,
    label="± 1 standard error",
)
plt.axvline(best_lasso_lambda, color="#E45756", linestyle="--", label=f"selected λ = {best_lasso_lambda:.3g}")
plt.xlabel(r"$\lambda$")
plt.ylabel("Cross-validated MSE")
plt.title("Cross-validation balances Lasso fit and sparsity")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Selected Lasso lambda: {best_lasso_lambda:.4g}")
print(f"Selected genes: {len(selected_genes)} / {n_genes}")
print(f"Direct signal genes recovered: {len(recovered_direct_genes)} / {len(true_gene_indices)}")
print("Selected gene indices:", selected_genes.tolist())

# Part 4 — Compare Prediction, Stability, and Sparsity

The three methods answer slightly different questions:

- **OLS:** Which coefficient vector minimizes training error, without controlling complexity?
- **Ridge:** Which small, stable coefficient vector predicts well when effects may be distributed across correlated genes?
- **Lasso:** Which sparse coefficient vector predicts well using a limited set of genes?

Only now do we evaluate the untouched test patients. For the centered response, predictions are $X\hat\beta$. On the original response scale, we would add the training mean $\bar y_{\text{train}}$ back to every prediction.

In [ ]:
# Part 4.1 — Final evaluation on the untouched test set
# ============================================================

models = {
    "OLS": ols_coef,
    "Ridge": ridge_coef,
    "Lasso": lasso_coef,
}

comparison_rows = []
for model_name, coefficients in models.items():
    comparison_rows.append({
        "Model": model_name,
        "Train RMSE": rmse(y_train, X_train @ coefficients),
        "Test RMSE": rmse(y_test, X_test @ coefficients),
        "L2 coefficient norm": np.linalg.norm(coefficients),
        "Nonzero coefficients": int(np.sum(np.abs(coefficients) > 1e-8)),
    })

comparison = pd.DataFrame(comparison_rows).set_index("Model")
display(comparison.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
comparison[["Train RMSE", "Test RMSE"]].plot.bar(ax=axes[0], rot=0, color=["#4C78A8", "#F58518"])
axes[0].set_title("Training fit versus generalization")
axes[0].set_ylabel("RMSE (lower is better)")

axes[1].bar(comparison.index, comparison["Nonzero coefficients"], color=["#9D755D", "#4C78A8", "#54A24B"])
axes[1].set_title("Model sparsity")
axes[1].set_ylabel("Number of nonzero coefficients")

plt.tight_layout()
plt.show()

best_test_model = comparison["Test RMSE"].idxmin()
print(f"Lowest test RMSE in this simulation: {best_test_model}")

assert np.sum(np.abs(lasso_coef) > 1e-8) < np.sum(np.abs(ridge_coef) > 1e-8)
print("Checkpoint passed: Lasso is sparser than Ridge.")

In [ ]:
# Part 4.2 — Inspect biological interpretation carefully
# ============================================================
# Compare estimated coefficients with the known simulation truth. In real
# studies, true_beta is unknown, which makes biological interpretation harder.

gene_table = pd.DataFrame({
    "Gene": [f"Gene_{j:03d}" for j in range(n_genes)],
    "True coefficient": true_beta,
    "Ridge coefficient": ridge_coef,
    "Lasso coefficient": lasso_coef,
    "Direct signal gene": np.isin(np.arange(n_genes), true_gene_indices),
})

# Show the largest Lasso coefficients because these genes would attract
# attention in a real biomarker analysis.
top_lasso = gene_table.iloc[np.argsort(np.abs(lasso_coef))[-15:][::-1]]
display(top_lasso.reset_index(drop=True).round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(true_beta, ridge_coef, alpha=0.65, color="#4C78A8")
axes[0].set(xlabel="True coefficient", ylabel="Estimated coefficient", title="Ridge: dense shrinkage")

axes[1].scatter(true_beta, lasso_coef, alpha=0.65, color="#54A24B")
axes[1].set(xlabel="True coefficient", ylabel="Estimated coefficient", title="Lasso: sparse selection")

for ax in axes:
    ax.axhline(0, color="black", linewidth=0.7)
    ax.axvline(0, color="black", linewidth=0.7)

plt.tight_layout()
plt.show()

proxy_genes = np.setdiff1d(selected_genes, true_gene_indices)
print("Selected correlated proxies rather than direct signal genes:", proxy_genes.tolist())
print("A predictive selected gene is not automatically a causal biomarker.")

# Part 5 — Summary and Short Exercises

## Main Findings

1. **High dimensionality creates ambiguity.** When $p>n$, many coefficient vectors can explain the training observations.
2. **Training error is insufficient.** OLS can fit the training patients closely yet generalize poorly.
3. **Ridge controls variance.** Its $L_2$ penalty shrinks correlated coefficients smoothly and usually retains every gene.
4. **Lasso produces sparsity.** Its $L_1$ penalty can set coefficients exactly to zero, but correlated genes may substitute for one another.
5. **Cross-validation selects complexity.** The test set must remain untouched until the final evaluation.
6. **Prediction is not causation.** A selected gene may be a correlated proxy, and selection can change across patient cohorts.

## Short Exercises

1. Increase `noise_sd` from `1.5` to `3.0`. Which method's test error changes the most?
2. Change the gene-specific noise from `0.35` to `0.10`, making pathway genes more strongly correlated. Does Lasso select the same genes?
3. Reduce `n_samples` from `120` to `80`. How do prediction error and feature recovery change?

## What We Deliberately Did Not Cover

To keep the tutorial focused, we did not include Elastic Net, logistic regression, nested cross-validation, formal statistical inference after feature selection, or a real patient dataset. Elastic Net is a natural next step when groups of correlated genes should enter the model together.

## References

1. Hoerl, A. E. & Kennard, R. W. **Ridge Regression: Biased Estimation for Nonorthogonal Problems.** *Technometrics* 12, 55–67 (1970). https://doi.org/10.1080/00401706.1970.10488634
2. Tibshirani, R. **Regression Shrinkage and Selection via the Lasso.** *Journal of the Royal Statistical Society: Series B* 58, 267–288 (1996). https://doi.org/10.1111/j.2517-6161.1996.tb02080.x
3. Hastie, T., Tibshirani, R. & Friedman, J. **The Elements of Statistical Learning.** Springer (2009). https://hastie.su.domains/ElemStatLearn/

> Reproducibility note: every random operation uses `SEED = 42`. Changing the seed represents sampling a different hypothetical patient cohort and may change which correlated genes Lasso selects.